# DAX 40 AI-Analyse — Siemens Proof of Concept
**Masterarbeit: Data Availability & KI-Wahrnehmung im Zeitverlauf**

Dieses Notebook:
1. Lädt den vorbereiteten Corpus
2. Führt N-Gram-Analysen durch
3. Erstellt Charts für den Zeitverlauf

To be added:
1. Stupidity Check: Damit z.B. "Daten" nicht in falschem Kontext erfasst wird
2. n-grams wie Chatbot, LLM hinzufügen
3. Manche Reden werden nicht erkannt (0 Wörter), obwohl sie richtig abgelegt sind. Außerdem sicherstellen, ob Reden vollständig erkannt werden.
4. Verhältnis KI zu Daten Begriffen --> was ist der Zusammenhang
5. n-grams zu Data analyiseren ("Wordcloud"). Wird "Datensilo", "Datenverfügbarkeit" o.Ä. häufig erwähnt
6. Suche ebenfalls nach n-grams für KI Strategie oder Daten Strategie oder nach Sätzen, in denen KI/Daten und Strategie zusammen vorkommt --> wird das Thema gesondert strategisch betrachtet?
7. Analysen nach Branche
8. Companies mentioning AI and Data in their speeches (of companies mentioning AI/of all companies)
9. Häufigkeit der Erwähnungen (# von 0, # von 1, # von 2-4, # von > 5)

In [ ]:
# Setup & Imports
import sys
sys.path.append('..')  # Damit 'scripts' Module importierbar sind

import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Eigene Module
from scripts.analyzer import (
    load_config, load_corpus,
    count_term, compute_all_frequencies,
    plot_term_over_time, plot_heatmap_terms_years,
    plot_doc_type_comparison
)

# Konfiguration laden
config = load_config()
print('Konfiguration geladen ✓')
print(f'Unternehmen: {list(config["companies"].keys())}')
print(f'Analyse-Terme: {config["ngrams"]["terms_1gram"]}') 

## 1. Corpus laden & Überblick

In [ ]:
# Corpus laden (nach Extraktion verfügbar)
corpus = load_corpus(config)
print(f'Corpus: {len(corpus)} Einträge')
print(f'Spalten: {list(corpus.columns)}')
corpus[['company', 'year', 'wordcount_annual_report', 'wordcount_ceo_speech', 
        'wordcount_supervisory_speech', 'wordcount_total']]

In [ ]:
# Verfügbarkeit der Dokumente
availability_cols = [c for c in corpus.columns if c.startswith('available_')]
corpus[['company', 'year'] + availability_cols].style.background_gradient(
    cmap='RdYlGn', subset=availability_cols
)

## 2. N-Gram Frequenzen berechnen

In [ ]:
# Alle Frequenzen berechnen
freq_df = compute_all_frequencies(corpus, config)
print(f'Frequenz-DataFrame: {len(freq_df)} Zeilen')
freq_df.head(20)

In [ ]:
# Pivot-Tabelle: Terme × Jahre für Siemens Geschäftsbericht
siemens_ar = freq_df[
    (freq_df['company'] == 'siemens') & 
    (freq_df['doc_type'] == 'annual_report')
].pivot_table(
    index='term', 
    columns='year', 
    values='count_normalized',
    aggfunc='mean'
).round(3)

# Farbige Darstellung
siemens_ar.style.background_gradient(cmap='YlOrRd', axis=None)

## 3. Kern-Visualisierungen

In [ ]:
# Chart 1: Kernterme im Zeitverlauf – Geschäftsbericht
fig = plot_term_over_time(
    freq_df,
    terms=['data', 'daten', 'ai', 'ki'],
    company='siemens',
    doc_type='annual_report',
    save_path=Path('../outputs/figures/siemens/01_core_terms_annual_report.png')
)
plt.show()

In [ ]:
# Chart 2: Heatmap alle Terme
all_terms = config['ngrams']['terms_1gram'] + config['ngrams']['terms_2gram']

fig = plot_heatmap_terms_years(
    freq_df,
    terms=all_terms,
    company='siemens',
    doc_type='annual_report',
    save_path=Path('../outputs/figures/siemens/02_heatmap_all_terms.png')
)
plt.show()

In [ ]:
# Chart 3: 'AI' nach Dokumenttyp – sieht CEO anders als Bericht?
for term in ['ai', 'ki']:
    fig = plot_doc_type_comparison(
        freq_df,
        term=term,
        company='siemens',
        save_path=Path(f'../outputs/figures/siemens/03_doctype_comparison_{term}.png')
    )
plt.show()

## 4. Interaktive Analyse (Plotly)

In [ ]:
# Interaktiver Zeitverlauf mit Plotly
siemens_data = freq_df[
    (freq_df['company'] == 'siemens') &
    (freq_df['doc_type'] == 'annual_report') &
    (freq_df['term'].isin(['data', 'daten', 'ai', 'ki']))
].sort_values('year')

fig = px.line(
    siemens_data,
    x='year',
    y='count_normalized',
    color='term',
    markers=True,
    title='Siemens – KI/Daten-Begriffe im Zeitverlauf (Geschäftsbericht)',
    labels={
        'count_normalized': 'Vorkommen pro 1.000 Wörter',
        'year': 'Jahr',
        'term': 'Begriff'
    },
    color_discrete_map={
        'data': '#5B5EA6', 'daten': '#9B2335',
        'ai': '#E87722', 'ki': '#2BAE66'
    }
)

# ChatGPT-Launch markieren
fig.add_vline(x=2022.9, line_dash='dash', line_color='gray', opacity=0.6,
              annotation_text='ChatGPT Launch', annotation_position='top right')

fig.update_layout(hovermode='x unified', template='simple_white')
fig.show()

## 5. Quick-Analyse: Rohe Zahlentabelle

In [ ]:
# Exportierbare Tabelle: Alle Zahlen für die Masterarbeit
export_table = freq_df[
    (freq_df['company'] == 'siemens') &
    (freq_df['doc_type'].isin(['annual_report', 'ceo_speech', 'supervisory_speech']))
].pivot_table(
    index=['term', 'doc_type'],
    columns='year',
    values=['count_raw', 'count_normalized'],
    aggfunc='sum'
).round(3)

# Exportieren
export_path = Path('../outputs/reports/siemens_ngram_table.xlsx')
export_table.to_excel(export_path)
print(f'Exportiert nach: {export_path}')
export_table